In [0]:
# Importar utils

In [0]:
%run ../src/utils


In [0]:
# -------------------------
# 1. Leer datos desde capa silver
# -------------------------
resumen_diario = spark.table("silver.resumen_diario")

log("Datos cargados desde capa silver")

# -------------------------
# 2. Validaciones de calidad
# -------------------------
validar_df(resumen_diario, "resumen_diario")

# Validación adicional: evitar valores nulos en métricas clave
from pyspark.sql.functions import col

nulls = resumen_diario.filter(
    col("ventas_totales").isNull()
).count()

if nulls > 0:
    raise Exception("Se encontraron valores nulos en ventas_totales")

log("Validaciones de calidad superadas")

# -------------------------
# 3. Escritura en capa gold (tabla final)
# -------------------------
resumen_diario.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.ventas_diarias")

log("Datos guardados en capa gold")

# -------------------------
# 4. Verificación final
# -------------------------
df_final = spark.table("gold.ventas_diarias")

log(f"Total registros en gold: {df_final.count()}")

df_final.show()

[LOG] Datos cargados desde capa silver
[INFO] resumen_diario tiene 2 registros
[LOG] Validaciones de calidad superadas
[LOG] Datos guardados en capa gold
[LOG] Total registros en gold: 2
+----------+--------------+---------------+---------+---------------+
|     fecha|ventas_totales|clientes_unicos|ventas_7d|ventas_prev_dia|
+----------+--------------+---------------+---------+---------------+
|2024-01-01|           300|              2|    300.0|           NULL|
|2024-01-02|           150|              1|    225.0|            300|
+----------+--------------+---------------+---------+---------------+

